## Reasoning Patch

In [1]:
%load_ext autoreload
%autoreload 2

### Overview

Patches whole attention layers (all heads at once) between a base and source prompt, sweeping a small window of consecutive layers together, restricted to the last-token (prediction) position.

### Set-up

Imports `_config`/`_dataset`/`_prompt`/`_mapping` plus the intervention primitives used for the layer sweep below.

In [ ]:
import torch
import gc
from tqdm import tqdm

import sys
sys.path.append("src")
import _config
import _dataset
import _prompt
import _mapping
from _intervention import prepare_batch_multitoken_intervention, batch_intervene

## Experiment Config

In [3]:
prompt_config = _config.PromptConfig(
    model_type="GPT-OSS_stepwise", # GPT-OSS or R1
    prompt_type="h_pre_penultimate_sum", # {null / h / h1 / h2}_{null / pre_result / pre_final_sum / ...}
)
intervention_config = _config.InterventionConfig(
    intervention_loc="3-layerwise", # restatement or reasoning or restatement_and_reasoning
    intervention_ids=[20],
    tok_pos_fn=_mapping.intervene_id_to_tok_pos_stepwise_3_digit_h,
    module_format="model.layers.{layer}.self_attn",
    pre_hook=False,
)
attention_config = _config.AttentionFreezeConfig(
    enabled=False,
    dataset_fn=_dataset.create_h_dataset,
    num_digits=3,
    prompt_fn=_prompt.get_stepwise_prompt,
    divide_num=22,
)
run_config = _config.RunConfig(
    experiment_root="experiments/attention_heads",
    result_dir="layer",
    filename_order="prompt_location",
    use_prefixed_location_in_filename=True,
)

intervention_ids = _config.resolve_intervention_ids(
    prompt_config.model_type,
    prompt_config.prompt_type,
    intervention_config.intervention_loc,
    intervention_config.intervention_ids,
)
# Only the last-token position is patched: that is the position whose attention-
# head output feeds directly into the next-token prediction.
tok_pos_list = [intervention_config.tok_pos_fn[attention_config.divide_num] - 1]
modifier_fn = lambda prompt, add_ds_entry: prompt

## Set up Experiment

Loads the model/tokenizer, builds the (here disabled) attention-freeze hooks, and loads the configured base/source prompt pairs to patch.

In [4]:
model, tokenizer = _config.load_model(prompt_config.model_type)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [5]:
def intervene_on_final_sum(prompt, add_ds_entry):
    source_prompt = attention_config.prompt_fn(
        add_ds_entry["source_1_digits"],
        add_ds_entry["source_2_digits"],
        add_ds_entry["source_1_num"],
        add_ds_entry["source_2_num"],
    )
    return _prompt.get_intervened_prompt(intervention_ids, prompt, source_prompt)

modifier_fn = intervene_on_final_sum

In [6]:
attention_freeze_hooks = _config.build_attention_freeze_hooks(
    model,
    tokenizer,
    attention_config,
    modifier_fn=modifier_fn,
)

In [7]:
prompts = _config.load_prompts(prompt_config)
print(f"loaded {len(prompts)} prompts")

loaded 256 prompts


In [8]:
print(intervention_ids)

[20]


In [9]:
# for i, row in prompts.iterrows():
#     print(list(enumerate(tokenizer.convert_ids_to_tokens(tokenizer(row["base_prompt"], add_special_tokens=False, return_tensors="pt")["input_ids"][0]))))

In [10]:
print(tok_pos_list)

[240]


## Run Experiment

For each layer, patches a small window of consecutive layers' attention output jointly at the last token, and records the effect on the factual vs counterfactual answer probability.

In [ ]:
batch_size = run_config.resolved_batch_size(attention_config.enabled)

header = list(prompts.columns) + ['intervention_ids', 'intervention_id', 'layer', 'generated_text', 'factual_label_probability', 'counterfactual_label_probability']
filepath = _config.build_output_filepath(prompt_config, intervention_config, run_config, header)

for i in tqdm(range(0, len(prompts), batch_size)):
    # Preparing prompts and labels
    batch_rows = prompts.iloc[i:i+batch_size]
    factual_labels, counterfactual_labels = _config.prepare_label_tensors(tokenizer, batch_rows)

    # Preparing intervention hooks
    for layer in range(len(model.model.layers)):
        intervene_hooks = []
        layers = range(layer, min(layer + 3, len(model.model.layers)))
        tokens, source_tokens, hooks = prepare_batch_multitoken_intervention(
            model,
            tokenizer,
            layers,
            [-1],
            batch_rows['base_prompt'].tolist(),
            batch_rows['source_prompt'].tolist(),
            module_format=intervention_config.module_format,
            pre_hook=intervention_config.pre_hook,
        )
        intervene_hooks += hooks
        input_length = tokens["input_ids"].shape[1]
        assert input_length == 241

        # Forward pass
        with torch.no_grad():
            output = batch_intervene(model, tokens["input_ids"], attention_freeze_hooks + intervene_hooks, attention_mask=tokens["attention_mask"], pre_hook=intervention_config.pre_hook)
        pred_toks = output.logits[:,-1,:].argmax(dim=-1)
        prob = torch.nn.functional.softmax(output.logits[:,-1,:], dim=-1)
        factual_prob = prob[torch.arange(prob.shape[0]), factual_labels]
        counterfactual_prob = prob[torch.arange(prob.shape[0]), counterfactual_labels]
        tokens["input_ids"] = torch.cat([tokens["input_ids"], pred_toks.unsqueeze(-1)], dim=1)
        del output
        
        # Writing results
        for j, (_, row) in enumerate(batch_rows.iterrows()):
            generated_text = tokenizer.decode(tokens["input_ids"][j,input_length:]).replace(tokenizer.pad_token[-1], "")
            _config.write_to_csv(filepath, row.to_list() + [intervention_ids, ', '.join(str(id) for id in intervention_ids), layer, generated_text.replace(tokenizer.pad_token[-1], ""), factual_prob[j].item(), counterfactual_prob[j].item()])

        del tokens, source_tokens, intervene_hooks
        torch.cuda.empty_cache()
        gc.collect()

 45%|██████████████████████████████████████████████████▍                                                            | 5/11 [21:56<26:18, 263.04s/it]